# 模块 3：导航纯算法示例

本 notebook 直接复用 `navigator.py`，展示航迹选择、`Waypoint`/`PositionCommand`、短时保持和 0.5 s 超时悬停。不导入 ROS 或 MAVROS 消息。

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while root != root.parent and not (root / 'code' / 'navigator.py').is_file():
    root = root.parent
assert (root / 'code' / 'navigator.py').is_file(), '未找到项目根目录'
sys.path.insert(0, str(root / 'code'))

from datatypes import Pose, Track
from navigator import on_timer, select_track, send_waypoint, waypoints_from_track

def track(track_id, position, age):
    return Track(track_id, position, (0.0, 0.0, 0.0), (0.0,) * 36, age)

## 1. 航迹选择与位置指令

选择 `age` 最大者，并在同龄时选 `id` 最小者。算法返回纯 `PositionCommand`。

In [ ]:
tracks = [
    track(8, (8.0, 1.0, 3.0), age=4),
    track(3, (3.0, -2.0, 5.0), age=9),
    track(1, (1.0, 7.0, 2.0), age=9),
]
selected = select_track(tracks)
waypoint = waypoints_from_track(selected)[0]
command = send_waypoint(waypoint)
print('selected track:', selected.id, 'age=', selected.age)
print('waypoint      :', waypoint)
print('command       :', command)
assert selected.id == 1
assert command.coordinate_frame == 1 and command.type_mask == 3576

## 2. 50 Hz 时钟层的三种输出

`on_timer` 本身不依赖时钟；节点以 50 Hz 调用它，并传入距最后非空输入的秒数。

In [ ]:
pose = Pose(position=(20.0, 30.0, 6.0), orientation=(0.0, 0.0, 0.0, 1.0))
fresh = on_timer(tracks, valid_age=0.0, last_cmd=None, current_pose=pose)
held = on_timer([], valid_age=0.30, last_cmd=fresh, current_pose=pose)
hover = on_timer([], valid_age=0.51, last_cmd=fresh, current_pose=pose)
print('fresh target :', fresh.position)
print('short hold   :', held.position)
print('timeout hover:', hover.position)
assert held == fresh
assert hover.position == pose.position
assert hover.coordinate_frame == 1 and hover.type_mask == 3576

## 3. 边界：无航迹且无可保持指令

此时直接根据最新 `Pose` 生成悬停指令。

In [ ]:
empty = on_timer([], valid_age=float('inf'), last_cmd=None, current_pose=pose)
print(empty)
assert empty.position == (20.0, 30.0, 6.0)